# Lang-Bridge — dub a folder of videos

Runs the whole pipeline — **fetch → transcribe → translate → voice → mix** — on a folder of
videos and/or YouTube links, from any language into any languages, with no app needed.
The Lang-Bridge app can open the results to check, fix and listen.

## Where it runs

| | Before **Run all** |
|---|---|
| **Kaggle** | Import this notebook (File → Import notebook → Link: `https://raw.githubusercontent.com/LetPastorSpeakInYourLanguage/lang_recomp/main/notebooks/lang_bridge.ipynb`); Settings → Accelerator **GPU T4 x2** (both are used), Internet **on** (phone-verified account). Use `STORAGE = bucket` to keep results. |
| **Colab** | Runtime → Change runtime type → **T4 GPU**. Drive is connected by itself when a setting points into `/content/drive/…`. |
| **Your own machine** | Jupyter with Python ≥ 3.10 (a fresh virtual environment is best). An NVIDIA GPU is used if there is one, otherwise the CPU (fine for transcripts, slow for voices). PyTorch and ffmpeg are installed if missing. |

Everything is set in the **Settings** cell, the token and bucket included.

Safe to stop at any time: run again with the same settings to continue (or see RUN_FOLDER).
Guide: [https://github.com/LetPastorSpeakInYourLanguage/lang_recomp/tree/main/notebooks](https://github.com/LetPastorSpeakInYourLanguage/lang_recomp/tree/main/notebooks)

In [ ]:
#@title Settings
STORAGE = "folder"  #@param ["folder", "bucket"]
#@markdown `folder`: results go to OUTPUT. `bucket`: results are pushed to BUCKET after every stage (the app syncs them), and VIDEOS may be a folder inside the bucket.
VIDEOS = ""  #@param {type:"string"}
#@markdown A folder of videos (subfolders are works; `name.srt` beside a video is used instead of transcribing). With STORAGE = bucket, a folder in the bucket (e.g. `videos/sermons`).
YOUTUBE = ""  #@param {type:"string"}
#@markdown YouTube video or playlist links, separated by spaces.
LANGUAGE = "en"  #@param {type:"string"}
#@markdown The videos' language (e.g. `en`, `tr`, `am`).
TARGETS = "am"  #@param {type:"string"}
#@markdown Languages to dub into, e.g. `am om ti` or `en`.
OUTPUT = ""  #@param {type:"string"}
#@markdown STORAGE = folder: where results go (a Drive folder on Colab, any folder on your machine). Empty: `lb-out` in WORK_DIR.
BUCKET = ""  #@param {type:"string"}
#@markdown A Hugging Face bucket, `namespace/name`. Needed for STORAGE = bucket; with STORAGE = folder it is a backup the results are also pushed to.
STAGES = "fetch transcribe translate voice mix"  #@param {type:"string"}
#@markdown Which stages: e.g. `fetch transcribe` for transcripts only.
DUB_LIMIT = 0  #@param {type:"integer"}
#@markdown Voice and mix only the first N videos (0 = all). The rest are transcribed and translated.
LIMIT = 0  #@param {type:"integer"}
#@markdown Work on only the first N videos (0 = all).
ASR_MODELS = "am=badrex/Ethio-ASR-amharic"  #@param {type:"string"}
#@markdown A speech recogniser per source language from Hugging Face (`lang=repo`, space separated). Languages not named use Whisper large-v3.
ALIGNERS = ""  #@param {type:"string"}
#@markdown Word aligners per language (`lang=repo`), over the built-in ones for ~27 languages.
CAPTIONS = 0  #@param {type:"integer"}
ALIGN_CAPTIONS = 0  #@param {type:"integer"}
#@markdown YouTube only: also take YouTube's captions for the first N videos, force-align the first M, and compare with Whisper.
RUN_FOLDER = ""  #@param {type:"string"}
#@markdown Continue a run: its folder (`…/runs/<run>`; with STORAGE = bucket, `runs/<run>`). Settings above are then ignored.

HF_TOKEN = ""  #@param {type:"string"}
#@markdown Your [Hugging Face token](https://huggingface.co/settings/tokens) (write access if you use a bucket). **Never share or commit this notebook with a token filled in.**
DEVICE = "auto"  #@param ["auto", "cpu", "1 GPU", "2 GPUs"]
#@markdown `auto`: every GPU there is (Kaggle T4 x2 → 2, Colab T4 → 1), else the CPU.
WORK_DIR = ""  #@param {type:"string"}
#@markdown Code, models and scratch files. Empty: `/kaggle/working`, `/content`, or `~/lang-bridge` on your own machine.
CODE_BRANCH = "main"  #@param {type:"string"}

In [ ]:
#@title Get Lang-Bridge
import importlib.util, io, os, shutil, subprocess, sys, urllib.request, zipfile
from pathlib import Path

def _runtime():
    if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
        return 'kaggle'
    try:
        if importlib.util.find_spec('google.colab'):
            return 'colab'
    except (ImportError, ValueError):
        pass
    return 'local'

RUNTIME = _runtime()
ROOT = Path(WORK_DIR).expanduser() if WORK_DIR.strip() else Path({'kaggle': '/kaggle/working', 'colab': '/content'}.get(RUNTIME) or Path.home() / 'lang-bridge')
ROOT.mkdir(parents=True, exist_ok=True)
BRANCH = CODE_BRANCH.strip() or 'main'
# On your own machine, a notebook opened from inside a Lang-Bridge checkout uses that checkout.
_here = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'worker' / 'lb_worker').is_dir() and (p / 'app').is_dir()), None)
if RUNTIME == 'local' and _here:
    CODE = _here  # as it is, on whatever branch it has checked out (CODE_BRANCH is not used)
    BRANCH = subprocess.run(['git', '-C', str(CODE), 'rev-parse', '--abbrev-ref', 'HEAD'], capture_output=True, text=True).stdout.strip() or 'checkout'
else:
    CODE = ROOT / 'lang_recomp'
    if shutil.which('git'):
        if not (CODE / '.git').exists():
            shutil.rmtree(CODE, ignore_errors=True)
            subprocess.run(['git', 'clone', '-q', '--depth', '1', '-b', BRANCH, 'https://github.com/LetPastorSpeakInYourLanguage/lang_recomp', str(CODE)], check=True)
        else:
            subprocess.run(['git', '-C', str(CODE), 'fetch', '-q', '--depth', '1', 'origin', BRANCH], check=True)
            subprocess.run(['git', '-C', str(CODE), 'checkout', '-q', '-f', '-B', BRANCH, 'FETCH_HEAD'], check=True)
    else:  # no git on this machine: the branch as a zip
        shutil.rmtree(CODE, ignore_errors=True)
        with urllib.request.urlopen('https://github.com/LetPastorSpeakInYourLanguage/lang_recomp/archive/refs/heads/' + BRANCH + '.zip') as r:
            zipfile.ZipFile(io.BytesIO(r.read())).extractall(ROOT)
        next(p for p in ROOT.glob('lang_recomp-*') if p.is_dir()).rename(CODE)
sys.path[:0] = [str(CODE), str(CODE / 'worker')]
_v = subprocess.run(['git', '-C', str(CODE), 'log', '-1', '--format=%h (%cd)'], capture_output=True, text=True).stdout.strip() if (CODE / '.git').exists() else 'zip'
print(f'Lang-Bridge {BRANCH} {_v} in {CODE} · running on {RUNTIME}')

In [ ]:
#@title Set up this machine
from lb_worker import env
M = env.setup(RUNTIME, ROOT, DEVICE, STORAGE, OUTPUT, BUCKET, HF_TOKEN, paths=(VIDEOS, RUN_FOLDER))

In [ ]:
#@title Run
from lb_worker.research import pull_folder, run_folder, run_manifest

pairs = lambda text: dict(p.split('=', 1) for p in text.split() if '=' in p)
videos = VIDEOS.strip() or None
in_bucket = lambda p: M['storage'] == 'bucket' and not os.path.isabs(p)
if videos and in_bucket(videos):
    videos = str(pull_folder(M['bucket'], videos, M['out'], M['token']))
if RUN_FOLDER.strip():
    folder = M['out'] / RUN_FOLDER.strip() if in_bucket(RUN_FOLDER.strip()) else RUN_FOLDER.strip()
    result = run_manifest(folder, hf_token=M['token'], bucket=M['bucket'])
else:
    result = run_folder(videos, M['out'], LANGUAGE.strip(), TARGETS.split(), STAGES.split(),
                        youtube=YOUTUBE.split() or None, dub_limit=DUB_LIMIT or None, limit=LIMIT or None,
                        captions=CAPTIONS, align_captions=ALIGN_CAPTIONS, hf_token=M['token'],
                        asr_models=pairs(ASR_MODELS), aligners=pairs(ALIGNERS), bucket=M['bucket'],
                        options=M['options'])

## Look at the results

- **Without the app**: the dubbed videos are in `library/<work>/<video>/export/` under the
  results folder (OUTPUT, or the bucket).
- **With a bucket**: in the Lang-Bridge app, Settings → Folders → add one of kind
  **Hugging Face bucket**, then **Sync**; the run's progress shows as it goes and its results
  open by themselves.
- **With a folder the app can see** (Drive for Desktop, or this PC): Settings → Folders → add
  it as a **results folder**; its runs appear under Home → Runs.

Done with the GPU? Colab: **Runtime → Disconnect and delete runtime**; Kaggle: **Stop session**.